In [1]:
import pandas as pd
import sqlite3
import os

df = pd.read_excel("/run/media/bharathy/DATA/Sales_Dataset_2024.xlsx")
df.columns = df.columns.str.replace(" ", "_").str.replace("-", "_")

os.makedirs("../data", exist_ok=True)
conn = sqlite3.connect("../data/superstore.db")
df.to_sql("orders", conn, if_exists="replace", index=False)
conn.close()

print("Loaded into superstore.db as table 'orders'")
print(df.shape)
print(df.columns.tolist())

Loaded into superstore.db as table 'orders'
(2000, 10)
['Date', 'Region', 'Product', 'Salesperson', 'Units_Sold', 'Unit_Price', 'Category', 'Revenue', 'Cost', 'Profit']


In [2]:
from google import genai
from dotenv import load_dotenv
import os
import re

load_dotenv(dotenv_path="../.env")
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

SCHEMA = """
Table: orders
Columns:
- Date (datetime) — order date
- Region (text) — North/South/East/West
- Product (text) — product name e.g. Smartwatch, Monitor, Mobile, Headphones
- Salesperson (text) — name of salesperson
- Units_Sold (float)
- Unit_Price (float)
- Category (text) — Accessories/Office/Electronics
- Revenue (float)
- Cost (float)
- Profit (float)
"""

print("Setup complete")

Setup complete


In [3]:
def generate_sql(question: str, error_context: str = None) -> str:
    error_note = f"\n\nYour previous attempt failed with this error: {error_context}\nFix the query." if error_context else ""
    
    prompt = f"""You are a SQLite expert. Given this table schema:

{SCHEMA}

IMPORTANT: This is SQLite, not PostgreSQL or MySQL. Use SQLite syntax only.
- For dates, use strftime('%Y', Date), strftime('%m', Date), or strftime('%Y-%m', Date)

Write a single valid SQLite SELECT query to answer this question:
"{question}"

Rules:
- Only output the raw SQL query, nothing else — no markdown, no explanation, no backticks
- Only use SELECT statements — never DROP, DELETE, UPDATE, INSERT, or ALTER
- Use the exact column and table names given above{error_note}
"""
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    raw = response.text.strip()

    # Strip markdown code fences (handles ```sql ... ``` in any position)
    raw = re.sub(r"^```(?:sql)?\s*|\s*```$", "", raw.strip(), flags=re.MULTILINE).strip()

    return raw

print("generate_sql defined")

generate_sql defined


In [4]:
def run_query(sql: str) -> pd.DataFrame:
    sql = sql.strip()
    
    if not sql.upper().startswith("SELECT"):
        raise ValueError(f"Only SELECT queries are allowed. Got: {sql[:50]}")
    
    conn = sqlite3.connect("../data/superstore.db")
    result = pd.read_sql(sql, conn)
    conn.close()
    return result

print("run_query defined")

run_query defined


In [5]:
def ask(question: str, max_retries: int = 2):
    error_context = None
    sql = None
    for attempt in range(max_retries + 1):
        sql = generate_sql(question, error_context)
        try:
            result = run_query(sql)
            return sql, result
        except Exception as e:
            error_context = str(e)
            print(f"Attempt {attempt+1} failed: {error_context}")
    raise RuntimeError(f"Failed after {max_retries+1} attempts. Last SQL:\n{sql}")

print("ask defined")

ask defined


In [6]:
sql, result = ask("Which region had the lowest monthly revenue and which month?")
print("Final SQL:", sql)
result

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Final SQL: SELECT Region, strftime('%Y-%m', Date) AS Month, SUM(Revenue) AS Total_Revenue FROM orders GROUP BY Region, Month ORDER BY Total_Revenue ASC LIMIT 1


,Region,Month,Total_Revenue
0,Easst,2024-03,1429.0
